# Faisons apprendre notre premier modèle

Réunir les briques déjà comprises dans une boucle sur trois exemples.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#premiere-boucle-minimale).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous pouvons enfin revenir à B. Nous ne modifierons plus son score directement : l’optimiseur ajustera la règle commune aux trois extraits. Retrouvons X, cibles, les poids [[0,1],[0,−1]] et les biais nuls. La première perte moyenne vaut toujours environ 3,02.

Notre modèle ne contient encore qu’une couche. Pour donner un nom à l’ensemble et retrouver ses paramètres, organisons ce calcul dans un nn.Module. Il ne crée aucune nouvelle règle mathématique.

__init__ prépare l’objet ; super().__init__ initialise la partie PyTorch qui enregistre les couches. self désigne cette instance du modèle ; self.linear conserve notre Linear. forward décrit ensuite le calcul d’un appel : modele(X) utilise cette méthode. Nous n’appelons pas forward directement, afin de garder les mécanismes de nn.Module.

modele = Classifieur() crée l’objet. Le programme complet installe les poids connus, puis modele.parameters() les fournit avec les biais à SGD. Les paramètres d’une couche sont suivis par autograd par défaut : nous n’ajoutons donc pas requires_grad=True ici.

Sans reduction="none", ce critère produit directement la perte moyenne. Le taux reste 0,1. À chaque tour, nous refaisons les scores avec les paramètres courants, calculons le coût, dérivons, puis appliquons un pas.

Le tableau tient entièrement en mémoire : un tour traite les trois exemples. C’est donc aussi une époque sur ce petit jeu. range(201) donne les indices 0 à 200. Les gradients s’additionnent par défaut : zero_grad efface les contributions précédentes avant le prochain backward.

Repérez la seule ligne qui modifie les paramètres. Si elle disparaissait, répéter les autres instructions 201 fois ne suffirait pas à réparer B. Prévoyez les décisions que l’on retrouverait dans ce cas.

Dans le programme réel, nous gardons step. historique.append(perte.item()) mémorise un nombre Python pour suivre le coût, sans remplacer la perte utilisée par backward. Les valeurs ci-dessous sont lues avant le pas de leur tour.

Après le dernier step, nous devons encore recalculer les scores pour observer les derniers paramètres. Le contexte no_grad évite alors de préparer une nouvelle dérivation : cette passe sert seulement à lire le résultat.

B est maintenant classée comme percussion, sans qu’on ait retouché sa case de score. Cela prouve que la boucle agit sur la règle. Pour savoir ce qu’elle fait au-delà de ces trois lignes apprises, il nous faut maintenant des exemples réservés.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

X et cibles conservent les valeurs, les axes et les types du départ.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Définir la règle avant de l’utiliser

super prépare l’enregistrement PyTorch ; self.linear contient la couche ; forward l’utilise à chaque appel du modèle.

In [6]:
class Classifieur(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, mesures):
        return self.linear(mesures)


### Calculer l’état suivant

Nous instancions le module, puis remplaçons l’initialisation par les poids et biais connus.

In [7]:
modele = Classifieur()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
with torch.no_grad():
    modele.linear.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.linear.bias.zero_()


### Calculer l’état suivant

Le critère moyenne les trois coûts. SGD reçoit les paramètres enregistrés ; historique accueillera les observations.

In [9]:
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
historique = []


In [10]:
observer('historique', historique)

historique = []


### Répéter la même étape sur les données

Chaque tour remet les gradients à zéro puis relit le même lot avec les paramètres actuels.

Le coût lu avec item a été calculé avant step. Le journal n’est pas une nouvelle perte évaluée après la mise à jour.

In [11]:
for epoque in range(201):
    optimiseur.zero_grad()
    scores = modele(X)
    perte = critere(scores, cibles)
    perte.backward()
    optimiseur.step()
    historique.append(perte.item())
    if epoque in [0, 50, 200]:
        print("tour", epoque, "perte avant pas :", round(perte.item(), 3))


tour 0 perte avant pas : 3.018
tour 50 perte avant pas : 0.514
tour 200 perte avant pas : 0.226


### Appliquer le contexte explicite

La nouvelle passe utilise les paramètres après les 201 modifications. Elle ne prépare pas un autre backward.

In [12]:
with torch.no_grad():
    scores_finaux = modele(X)
    predictions = scores_finaux.argmax(dim=1)
    perte_finale = critere(scores_finaux, cibles)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("decisions finales :", predictions.tolist())
print("perte apres les 201 pas :", round(perte_finale.item(), 3))

decisions finales : [0, 1, 0]
perte apres les 201 pas : 0.225


In [14]:
assert bool(len(historique) == 201 and predictions.tolist() == [0, 1, 0]), 'premiere-boucle-minimale/main: len(historique) == 201 and predictions.tolist() == [0, 1, 0]'
assert bool(abs(historique[0] - 3.018476) < 2e-6 and abs(historique[50] - 0.514087) < 2e-6), 'premiere-boucle-minimale/main: abs(historique[0] - 3.018476) < 2e-6 and abs(historique[50] - 0.514087) < 2e-6'
assert bool(abs(historique[200] - 0.226087) < 2e-6 and abs(perte_finale.item() - 0.225241) < 2e-6), 'premiere-boucle-minimale/main: abs(historique[200] - 0.226087) < 2e-6 and abs(perte_finale.item() - 0.225241) < 2e-6'
assert bool(isinstance(modele, nn.Module) and len(list(modele.parameters())) == 2), 'premiere-boucle-minimale/main: isinstance(modele, nn.Module) and len(list(modele.parameters())) == 2'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Diagnostiquez deux itérations sur w² sans zero_grad : quel poids final depuis w=1, avec lr=0,1 ?

In [15]:
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le premier gradient vaut 2. Le second, 1,6, s’ajoute au précédent.

</details>

### Exercice 2

Nous remettons cette fois les gradients à zéro au début de chaque tour. Prévoyez le poids final.

In [16]:
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le premier poids devient 0,8. Le second pas utilise seulement le gradient courant, 1,6.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = 0.44
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = 0.64
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [19]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()